# 🏥 VoiceAid Health — CPU Backend

**CPU-only version** — No GPU required. Works on free Colab or your local machine.

## Key Differences from GPU Version
| Feature | GPU Version | CPU Version |
|---------|------------|-------------|
| ASR Model (Twi) | whisper-small (fine-tuned) | whisper-small (fine-tuned) |
| ASR Model (English) | whisper-medium | whisper-small (faster on CPU) |
| Intent Predictor | Qwen 2.5 1.5B LLM | ❌ Removed (too slow on CPU) |
| TTS | MMS VITS (GPU) | MMS VITS (CPU) |
| Transcription speed | ~1-3s per chunk | ~8-20s per chunk |

> ⚠️ **CPU mode is significantly slower.** Use the GPU notebook when Colab GPU is available.
> The `/predict/intent` endpoint is disabled — the app will fall back to local phrase suggestions.

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q fastapi "uvicorn[standard]" transformers torch pydub soundfile \
    python-multipart scipy nest-asyncio websockets accelerate
!killall cloudflared 2>/dev/null || true
!rm -f cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O cloudflared
!chmod +x cloudflared
print('✅ Dependencies installed!')

## Cell 2 — CPU Backend

### Section Notes
- **Section 1** — Same WebSocket Origin Bypass middleware (fixes 403 from Cloudflare)
- **Section 2** — Health routes (`/` and `/health`)
- **Section 3** — ASR Engine: uses `float32` + `device='cpu'`. English model switched to `whisper-small` for speed. Same `dysarthric_filter` and `ASR_GENERATE_KWARGS`
- **Section 4** — Same batch + streaming ASR routes
- **Section 5** — `/predict/intent` returns `501 Not Implemented` on CPU — the app falls back to local phrase chips automatically
- **Section 6** — Same MMS TTS engine on CPU
- **Section 7** — Same Cloudflare + Uvicorn deployment

In [ ]:
import io, re, time, json, base64, torch, numpy as np, scipy.io.wavfile, subprocess
from fastapi import FastAPI, UploadFile, File, Form, WebSocket, WebSocketDisconnect
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from starlette.types import ASGIApp, Receive, Scope, Send
from pydantic import BaseModel
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline, VitsModel, AutoTokenizer
import nest_asyncio, uvicorn

nest_asyncio.apply()

DEVICE = 'cpu'
DTYPE  = torch.float32   # float16 is not well-supported on CPU

# ============================================================
# SECTION 1: App Setup & Middleware
# ============================================================

app = FastAPI(title='VoiceAid Health CPU Backend')

class WebSocketOriginBypassMiddleware:
    """Fixes 403 Forbidden on WebSocket connections through Cloudflare tunnels."""
    def __init__(self, app: ASGIApp): self.app = app
    async def __call__(self, scope: Scope, receive: Receive, send: Send):
        if scope['type'] == 'websocket':
            headers = [(k, v) for k, v in scope.get('headers', []) if k.lower() != b'origin']
            headers.append((b'origin', b'http://localhost'))
            scope['headers'] = headers
        await self.app(scope, receive, send)

app.add_middleware(WebSocketOriginBypassMiddleware)
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_credentials=True,
                   allow_methods=['*'], allow_headers=['*'])

# ============================================================
# SECTION 2: Health Routes
# ============================================================

@app.get('/')
async def root():
    return {'message': 'VoiceAid Health CPU Backend is Online!', 'gpu': False, 'mode': 'cpu'}

@app.get('/health')
async def health():
    return {'status': 'healthy', 'gpu': False, 'mode': 'cpu'}

# ============================================================
# SECTION 3: ASR Engine (Whisper — CPU Mode)
# ============================================================
# Uses float32 (CPU-compatible). English model is whisper-small
# instead of whisper-medium for reasonable speed on CPU.

asr_pipes = {}

ASR_GENERATE_KWARGS = {
    'max_new_tokens': 128,
    'temperature': 0.0,
    'condition_on_prev_tokens': False,
    'no_speech_threshold': 0.6,
    'repetition_penalty': 1.3,
}

def load_asr(language='tw'):
    # Use whisper-small for English on CPU (whisper-medium is too slow)
    model_id = (
        'dennis-9/whisper-small_Akan_finetuned_v2' if language in ['tw', 'twi', 'akan']
        else 'openai/whisper-small'
    )
    if model_id in asr_pipes: return model_id
    print(f'\n🔥 Loading ASR ({model_id}) on CPU — this may take a minute...')
    model = AutoModelForSpeechSeq2Seq.from_pretrained(model_id, torch_dtype=DTYPE)
    processor = AutoProcessor.from_pretrained(model_id)
    asr_pipes[model_id] = pipeline(
        'automatic-speech-recognition', model=model,
        tokenizer=processor.tokenizer, feature_extractor=processor.feature_extractor,
        torch_dtype=DTYPE, device=DEVICE, chunk_length_s=30,
    )
    print(f'✅ ASR Loaded: {model_id}')
    return model_id

def dysarthric_filter(text: str) -> str:
    text = re.sub(r'\b(.+?)(?:\s+\1\b)+', r'\1', text, flags=re.IGNORECASE)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    return text.strip()

# ============================================================
# SECTION 4: ASR Routes
# ============================================================

@app.post('/asr/transcribe')
async def transcribe(file: UploadFile = File(...), language: str = Form('tw')):
    from pydub import AudioSegment
    audio = AudioSegment.from_file(io.BytesIO(await file.read()))
    audio = audio.set_channels(1).set_frame_rate(16000)
    samples = np.array(audio.get_array_of_samples()).astype(np.float32) / 32768.0
    model_id = load_asr(language)
    result = asr_pipes[model_id](samples, generate_kwargs=ASR_GENERATE_KWARGS)
    return {'text': dysarthric_filter(result['text']), 'model': model_id, 'language': language}

@app.websocket('/asr/stream')
async def stream_transcription(websocket: WebSocket):
    await websocket.accept()
    print('✅ WebSocket client connected (CPU mode — expect slower responses)')
    try:
        while True:
            data = await websocket.receive_text()
            message = json.loads(data)
            audio_b64 = message.get('audio')
            language  = message.get('language', 'tw')
            chunk_id  = message.get('chunk_id', 0)
            if not audio_b64: continue

            from pydub import AudioSegment
            audio = (AudioSegment.from_file(io.BytesIO(base64.b64decode(audio_b64)))
                     .set_channels(1).set_frame_rate(16000))
            samples = np.array(audio.get_array_of_samples()).astype(np.float32) / 32768.0

            model_id = load_asr(language)
            result   = asr_pipes[model_id](samples, generate_kwargs=ASR_GENERATE_KWARGS)
            clean    = dysarthric_filter(result['text'])

            if not clean or len(clean.strip()) < 2:
                print(f'[CPU ASR] ⏭️ Skipping silent chunk {chunk_id}')
                continue

            await websocket.send_json({
                'text': clean, 'chunk_id': chunk_id,
                'model': model_id, 'is_final': False, 'language': language,
            })
    except WebSocketDisconnect:
        print('📴 WebSocket client disconnected.')

# ============================================================
# SECTION 5: Intent Predictor — Disabled on CPU
# ============================================================
# Qwen 2.5 LLM is too slow on CPU for real-time use (~60s per request).
# This endpoint returns 501 so the React Native app falls back automatically
# to its built-in local medical phrase suggestions.

class PredictRequest(BaseModel):
    text: str
    language: str = 'tw'

@app.post('/predict/intent')
async def predict_intent(req: PredictRequest):
    return JSONResponse(
        status_code=501,
        content={'error': 'Intent prediction is not available in CPU mode. App will use local suggestions.'}
    )

# ============================================================
# SECTION 6: TTS Engine (Meta MMS — CPU Mode)
# ============================================================

tts_models = {}

def load_tts(lang_code):
    if lang_code in tts_models: return tts_models[lang_code]
    model_id = 'facebook/mms-tts-aka' if lang_code == 'tw' else 'facebook/mms-tts-eng'
    print(f'\n🌟 Loading TTS ({model_id}) on CPU...')
    model     = VitsModel.from_pretrained(model_id)  # CPU — no .to('cuda')
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tts_models[lang_code] = (model, tokenizer)
    print(f'✅ TTS Loaded: {model_id}')
    return model, tokenizer

@app.post('/tts/synthesize')
@app.get('/tts/synthesize')
async def synthesize(request_or_text, language: str = 'tw'):
    text   = request_or_text.text if hasattr(request_or_text, 'text') else request_or_text
    lang_id = 'tw' if language in ['tw', 'twi', 'akan'] else 'eng'
    model, tokenizer = load_tts(lang_id)
    inputs = tokenizer(text, return_tensors='pt')
    with torch.no_grad():
        output = model(**inputs).waveform
    audio_np = output.squeeze().numpy()
    sr       = model.config.sampling_rate
    audio_np = np.pad(audio_np, (0, int(0.5 * sr)), mode='constant')
    audio_np = (audio_np * 32767.0).astype(np.int16)
    audio_io = io.BytesIO()
    scipy.io.wavfile.write(audio_io, sr, audio_np)
    audio_io.seek(0)
    return StreamingResponse(audio_io, media_type='audio/wav')

# ============================================================
# SECTION 7: Deployment — Cloudflare Tunnel + Uvicorn
# ============================================================

print('🌍 Spinning up Cloudflare Tunnel...')
subprocess.Popen(
    'nohup ./cloudflared tunnel --url http://127.0.0.1:8000 > cloudflare.log 2>&1 &',
    shell=True
)
time.sleep(10)

with open('cloudflare.log', 'r') as log:
    content = log.read()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
    if match:
        public_url = match.group(0)
        print('\n' + '=' * 70)
        print('✅ VOICEAID CPU BACKEND IS ONLINE!')
        print(f'👉 COPY THIS URL INTO constants/config.ts:  {public_url}')
        print('=' * 70 + '\n')
    else:
        print('⚠️ Tunnel URL not found. Check cloudflare.log.')

config = uvicorn.Config(app, host='127.0.0.1', port=8000, ws='websockets')
server = uvicorn.Server(config)
await server.serve()

## API Reference

| Method | Endpoint | CPU Status |
|--------|----------|------------|
| `GET` | `/` | ✅ Fast |
| `GET` | `/health` | ✅ Fast |
| `POST` | `/asr/transcribe` | ⚠️ Slow (~10-20s) |
| `WebSocket` | `/asr/stream` | ⚠️ Slow (~8-20s per chunk) |
| `POST` | `/predict/intent` | ❌ Returns 501 (app uses local chips instead) |
| `GET/POST` | `/tts/synthesize` | ⚠️ Slow (~5-10s) |
| `GET` | `/docs` | ✅ Fast |

## When to Use Which Backend

| Situation | Use |
|-----------|-----|
| Colab GPU available (T4/A100) | `voiceaid_gpu_backend.ipynb` |
| Out of GPU compute units | `voiceaid_cpu_backend.ipynb` (this file) |
| Demo / testing locally | `voiceaid_cpu_backend.ipynb` |
| Live patient use / production | `voiceaid_gpu_backend.ipynb` |